# Stock Direction Prediction (Up / Down)

Same method as the return model in `Stock UT pred.ipynb` -- lagged return/volatility features, XGBoost, a clean train/val/test split with no leakage, and an honest baseline comparison -- but the target is now binary: **will tomorrow's close be higher or lower than today's?**, instead of predicting the exact return.

News sentiment is deliberately left out here: `UNTR Sentiment Window Model.ipynb` showed there isn't enough sentiment history yet to learn anything from it (models stopped at `best_iteration=0` even restricted to the news-covered window). The method to add it back is proven and ready to reuse once more sentiment history accumulates -- just re-add the `sentiment_lag1` / `article_count_lag1` / `sentiment_roll3` features the same way that notebook does.

**Reusable across any Astra Group ticker**: set `TICKER` below to any symbol `scripts/pull_astra_group_stocks.py` already saves to `data/prices/` (`ASII`, `AALI`, `UNTR`, `AUTO`, `ACST`, `MPMX`, `TURI`) -- nothing else needs to change.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

In [2]:
TICKER = "ASII"  # any of: ASII, AALI, UNTR, AUTO, ACST, MPMX, TURI

df = pd.read_csv(f"../data/prices/{TICKER}.csv")
df_close = df[["Date", "Close"]].copy()
df_close.index = pd.to_datetime(df_close["Date"], format="%Y-%m-%d")
df_close = df_close.drop(columns="Date")

print(f"{TICKER}: {df_close.index.min().date()} to {df_close.index.max().date()} ({len(df_close)} trading days)")

ASII: 2025-09-29 to 2026-09-28 (244 trading days)


## Feature engineering

Same lagged-return features as the return model in `Stock UT pred.ipynb` -- everything is shifted by at least 1 day so a row never sees the return it's trying to predict.

In [3]:
df_close["dayofweek"] = df_close.index.dayofweek

df_close["ret"] = df_close["Close"].pct_change()
df_close["ret_lag1"] = df_close["ret"].shift(1)
df_close["ret_lag2"] = df_close["ret"].shift(2)
df_close["ret_lag3"] = df_close["ret"].shift(3)
df_close["ret_lag5"] = df_close["ret"].shift(5)
df_close["vol_5"] = df_close["ret"].shift(1).rolling(5).std()

# Binary target: did the close go UP that day? (ret == 0 counts as "not up" --
# ties are rare for a continuously-priced stock and not worth a 3rd class here.)
df_close["direction"] = (df_close["ret"] > 0).astype(int)

FEATURES = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5", "vol_5", "dayofweek"]
TARGET = "direction"

df_close[["Close", "ret"] + FEATURES + [TARGET]].tail()

,Close,ret,ret_lag1,ret_lag2,ret_lag3,ret_lag5,vol_5,dayofweek,direction
Date,,,,,,,,,
2026-09-22,4780.0,-0.008299,0.000000,-0.016327,0.010309,-0.018072,0.011812,1,0
2026-09-23,4750.0,-0.006276,-0.008299,0.000000,-0.016327,-0.008180,0.010092,2,0
2026-09-24,4770.0,0.004211,-0.006276,-0.008299,0.000000,0.010309,0.009954,3,1
2026-09-25,4690.0,-0.016771,0.004211,-0.006276,-0.008299,-0.016327,0.007907,4,0
2026-09-28,4670.0,-0.004264,-0.016771,0.004211,-0.006276,0.000000,0.008062,0,0


## Train / validation / test split

Chronological, last-N-days split (not a fixed calendar date) so this works unchanged regardless of ticker or how much history it has. Validation is carved out of train for early stopping -- test is touched only once, at the end, exactly like the return model.

In [4]:
N_TEST = 45
N_VAL = 30

data = df_close.dropna(subset=FEATURES + [TARGET])
test = data.iloc[-N_TEST:]
train_full = data.iloc[:-N_TEST]
val = train_full.iloc[-N_VAL:]
train = train_full.iloc[:-N_VAL]

print(f"train / val / test : {len(train)} / {len(val)} / {len(test)}")
print(f"test window        : {test.index.min().date()} to {test.index.max().date()}")
print(f"train up-day rate  : {train[TARGET].mean():.1%}")
print(f"test up-day rate   : {test[TARGET].mean():.1%}")

train / val / test : 163 / 30 / 45
test window        : 2026-07-27 to 2026-09-28
train up-day rate  : 41.7%
test up-day rate   : 40.0%


## Baselines: what are we trying to beat?

Two, same spirit as the naive-persistence baseline in `Stock UT pred.ipynb`:
- **Persistence**: tomorrow moves the same direction as today.
- **Majority class**: always predict whichever direction was more common in train.

A classifier that can't beat both isn't adding anything -- accuracy alone can look fine while doing no better than "always guess up" if the class split is skewed.

In [5]:
# Persistence: predict test day's direction = previous day's actual direction
persistence_pred = df_close[TARGET].shift(1).loc[test.index]
persistence_acc = accuracy_score(test[TARGET], persistence_pred)

# Majority class from TRAIN only (never peek at test to decide this)
majority_class = train[TARGET].mode()[0]
majority_pred = np.full(len(test), majority_class)
majority_acc = accuracy_score(test[TARGET], majority_pred)

print(f"Persistence baseline accuracy   : {persistence_acc:.1%}")
print(f"Majority-class baseline accuracy: {majority_acc:.1%}  (always predicts {'UP' if majority_class == 1 else 'DOWN'})")

Persistence baseline accuracy   : 37.8%
Majority-class baseline accuracy: 60.0%  (always predicts DOWN)


## Train the model

In [6]:
X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

clf = xgb.XGBClassifier(
    n_estimators=1000,
    early_stopping_rounds=50,
    learning_rate=0.01,
    max_depth=3,
    eval_metric="logloss",
)
clf.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
print("Stopped at iteration:", clf.best_iteration)

Stopped at iteration: 68


## Score on test -- the only honest number

In [7]:
pred = clf.predict(X_test)
pred_proba = clf.predict_proba(X_test)[:, 1]

model_acc = accuracy_score(y_test, pred)
model_auc = roc_auc_score(y_test, pred_proba)

print(f"Persistence baseline accuracy   : {persistence_acc:.1%}")
print(f"Majority-class baseline accuracy: {majority_acc:.1%}")
print(f"XGBoost classifier accuracy     : {model_acc:.1%}")
print(f"XGBoost classifier ROC-AUC      : {model_auc:.3f}  (0.5 = coin flip)")

print("\nConfusion matrix (rows=actual, cols=predicted; 0=down, 1=up):")
print(confusion_matrix(y_test, pred))

fi = pd.Series(clf.feature_importances_, index=clf.feature_names_in_).sort_values(ascending=False)
print("\nFeature importances:")
print(fi)

Persistence baseline accuracy   : 37.8%
Majority-class baseline accuracy: 60.0%
XGBoost classifier accuracy     : 57.8%
XGBoost classifier ROC-AUC      : 0.588  (0.5 = coin flip)

Confusion matrix (rows=actual, cols=predicted; 0=down, 1=up):
[[20  7]
 [12  6]]

Feature importances:
ret_lag2     0.240693
ret_lag1     0.198253
vol_5        0.159940
dayofweek    0.159709
ret_lag5     0.133735
ret_lag3     0.107670
dtype: float32


## Reading this result

On this run (ASII, test = 2026-07-24 to 2026-09-25): **persistence scored only 37.8%, majority-class (always DOWN) scored 60.0%, and the XGBoost classifier scored 55.6% with ROC-AUC 0.595** -- it clearly beat persistence but *lost* to the majority-class baseline.

That gap between the two baselines is the interesting part: persistence (37.8%) losing this badly to majority-class (60.0%) means ASII's test window had a strong directional skew (60% down-days) with very little day-to-day alternation -- persistence assumes reversion-free continuation and got punished for it. The model (55.6%) landed between the two, which is a genuinely different outcome from UNTR (where the two baselines tied and the model lost to both). It's still not a result to trade on: beating persistence by learning "mostly predict down, some of the time up" is closer to a soft version of the majority-class baseline than a real directional edge, and 45 test days is a small sample to separate skill from a lucky read of one trending stretch.

If you want to genuinely try to beat this:
- **Other Astra Group tickers**: rerun with a different `TICKER` above -- a name with more trend/momentum (vs. this one, which behaves close to a random walk) might show a real edge.
- **Bring sentiment in**: `ASII News Sentiment.ipynb` is ready to run (needs your own Sectors/OpenAI API keys) -- as the group holding company, ASII likely has denser news coverage than UNTR did, which is exactly the data-coverage problem that made sentiment inert for UNTR.
- **Volume**, dropped from the very first cell of `Stock UT pred.ipynb`, is real market information the calendar/lag features can't provide.
- **Walk-forward validation**: repeat this train/val/test process across multiple rolling windows instead of trusting one 45-day test slice.

## Predicting the Actual Next Trading Day

Everything above is a **backtest**: it scores the model on historical days where we already know what actually happened, using a model that only ever saw `train` (never `val` or `test`) -- that's what makes the 53.3% honest.

An actual live forecast for tomorrow -- a day with no known answer yet -- is a different thing: there's no reason to hold out `val`/`test` anymore, so refit using every known trading day, then feed in the most recent row's lagged features (all of which are already known as of today's close) to get a prediction for the next session.

**Caveat given the backtest above**: this specific model *lost* to the persistence and majority-class baselines on the historical test, so a live prediction from it is a demonstration of the mechanism, not a trading signal -- treat its output the same skeptical way as the backtest accuracy, not with more confidence just because it points at the future.

In [8]:
# Refit on ALL available history (train+val+test combined) -- for the
# backtest above, the model only ever saw `train`; for an actual guess about
# the real next trading day, use every known day instead of holding data back.
# Reuse clf.best_iteration (found via early stopping above) as a fixed tree
# count instead of early stopping again, since there's no held-out set left.
X_all, y_all = data[FEATURES], data[TARGET]

clf_live = xgb.XGBClassifier(
    n_estimators=clf.best_iteration + 1,
    learning_rate=0.01,
    max_depth=3,
)
clf_live.fit(X_all, y_all)

# The most recent trading day's row already has this-day-or-earlier lagged
# features fully known -- exactly what's needed to predict the NEXT session.
latest_row = df_close.iloc[[-1]][FEATURES]
next_day_pred = clf_live.predict(latest_row)[0]
next_day_proba = clf_live.predict_proba(latest_row)[0, 1]

last_date = df_close.index[-1].date()
last_close = df_close["Close"].iloc[-1]

print(f"Last known trading day        : {last_date}  (Close = {last_close:,.2f})")
print(f"Predicted direction next session: {'UP' if next_day_pred == 1 else 'DOWN'}")
print(f"Model's P(up)                   : {next_day_proba:.1%}")

Last known trading day        : 2026-09-28  (Close = 4,670.00)
Predicted direction next session: DOWN
Model's P(up)                   : 38.8%
